## Create external location

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS gizmobox
URL 'abfss://project@dbdestorage.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL dbdeconn)

In [0]:
%fs
ls 'abfss://project@dbdestorage.dfs.core.windows.net/'

# CREATE CATALOG

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS gizmobox
MANAGED LOCATION 'abfss://project@dbdestorage.dfs.core.windows.net/'

In [0]:
%sql
USE CATALOG gizmobox;
CREATE SCHEMA landing
MANAGED LOCATION 'abfss://project@dbdestorage.dfs.core.windows.net/landing/';
CREATE SCHEMA bronze
MANAGED LOCATION 'abfss://project@dbdestorage.dfs.core.windows.net/bronze/';
CREATE SCHEMA silver
MANAGED LOCATION 'abfss://project@dbdestorage.dfs.core.windows.net/silver/';
CREATE SCHEMA gold
MANAGED LOCATION 'abfss://project@dbdestorage.dfs.core.windows.net/gold/';

In [0]:
%sql
DROP VOLUME IF EXISTS gizmobox.landing.v_gizmobox;

## CREATE VOLUME

In [0]:
%sql 
USE CATALOG gizmobox;
USE SCHEMA landing;
CREATE EXTERNAL VOLUME IF NOT EXISTS v_gizmobox
LOCATION 'abfss://project@dbdestorage.dfs.core.windows.net/operational_data/';

In [0]:
%fs
ls '/Volumes/gizmobox/landing/v_gizmobox/'

## Create Customers view

In [0]:
%sql
CREATE OR REPLACE VIEW gizmobox.bronze.v_customers AS
select  _metadata.file_path as file_path,* from JSON.`/Volumes/gizmobox/landing/v_gizmobox/customers/*`;

In [0]:
SELECT * FROM gizmobox.bronze.v_customers;

##Create Orders View

In [0]:
CREATE OR REPLACE VIEW gizmobox.bronze.v_orders AS
SELECT * FROM text.`/Volumes/gizmobox/landing/v_gizmobox/orders/*`;

In [0]:
SELECT * FROM gizmobox.bronze.v_orders;

## Create Addresses view

In [0]:
CREATE OR REPLACE VIEW gizmobox.bronze.v_addresses AS
SELECT * FROM read_files('/Volumes/gizmobox/landing/v_gizmobox/addresses/*', format => 'csv', sep => '\t')

In [0]:
SELECT * FROM gizmobox.bronze.v_addresses;

## Create memberships view

In [0]:
CREATE OR REPLACE VIEW gizmobox.bronze.v_memberships AS
SELECT * FROM binaryFile.`/Volumes/gizmobox/landing/v_gizmobox/memberships/*/*.png`

In [0]:
SELECT * FROM gizmobox.bronze.v_memberships;

In [0]:
%fs
ls '/Volumes/gizmobox/landing/v_gizmobox'

In [0]:
SELECT * FROM CSV.`abfss://project@dbdestorage.dfs.core.windows.net/payments/*`

In [0]:
DROP TABLE IF EXISTS gizmobox.bronze.payments;
CREATE TABLE gizmobox.bronze.payments
(
payment_id INTEGER,
order_id INTEGER,
payment_date TIMESTAMP,
payment_status STRING,
payment_method STRING
)
USING CSV
OPTIONS (
header = "true",
delimiter = ",")

LOCATION
"abfss://project@dbdestorage.dfs.core.windows.net/payments/";

In [0]:
SELECT * FROM gizmobox.bronze.payments;

## Create refunds table

In [0]:
CREATE TABLE gizmobox.bronze.refunds (  
    refund_id INT ,  
    payment_id INT NOT NULL,  
    refund_timestamp TIMESTAMP NOT NULL,  
    refund_amount DECIMAL(10, 2) NOT NULL,  
    refund_reason STRING NOT NULL
);

In [0]:
INSERT INTO gizmobox.bronze.refunds (refund_id, payment_id, refund_timestamp, refund_amount, refund_reason)  
VALUES  
(1, 66, '2025-01-10 11:30:00', 85.75, 'Payment Error:Retailer'),  
(2, 69, '2025-01-03 12:40:15', 120.50, 'Order Cancelled:Customer'),  
(3, 72, '2025-01-06 14:45:30', 65.00, 'Product Returned:Customer'),  
(4, 73, '2025-01-07 16:10:45', 210.99, 'Order Cancelled:Customer'),  
(5, 75, '2025-01-09 18:25:00', 45.20, 'Payment Error:Retailer'),  
(6, 80, '2025-01-10 09:35:20', 130.15, 'Order Cancelled:Customer'),  
(7, 83, '2025-01-12 11:20:40', 150.00, 'Product Returned:Customer'),  
(8, 85, '2025-01-14 13:15:30', 89.99, 'Order Cancelled:Customer'),  
(9, 89, '2025-01-15 15:00:00', 78.50, 'Payment Error:Retailer'),  
(10, 91, '2025-01-17 16:45:15', 250.75, 'Product Returned:Customer');

In [0]:
CREATE TABLE gizmobox.silver.payments
AS
SELECT payment_id,
       order_id,
       CAST(date_format(payment_timestamp,'yyyy-MM-dd') AS DATE) AS payment_date,
       date_format(payment_timestamp,'HH:mm:ss') AS payment_time,
       CASE payment_status
         WHEN 1 THEN 'Success'
         WHEN 2 THEN 'Pending'
         WHEN 3 THEN 'Cancelled'
         WHEN 4 THEN 'Failed'
       END AS payment_status,  
       payment_method
  FROM gizmobox.bronze.payments;